# Data preprocessing

In [ ]:
import pandas as pd

df1 = pd.read_csv('longmen_meteorological_data.csv')
df2 = pd.read_csv('spei_original.csv')
df1.info()
rows, columns = df1.shape
if rows < 100 and columns < 20:
    print('longmen_meteorological_data.csv all content：')
    print(df1.to_csv(sep='\t', na_rep='nan'))
else:
    print('longmen_meteorological_data.csv first rows content：')
    print(df1.head().to_csv(sep='\t', na_rep='nan'))
df2.info()
rows, columns = df2.shape
if rows < 100 and columns < 20:
    print('spei_original.csv all content：')
    print(df2.to_csv(sep='\t', na_rep='nan'))
else:
    print('spei_original.csv first rows content：')
    print(df2.head().to_csv(sep='\t', na_rep='nan'))
df1['time'] = pd.to_datetime(df1['time'])
df2['date'] = pd.to_datetime(df2['date'])
merged_df = pd.merge(df1, df2, left_on='time', right_on='date', how='left')
merged_df = merged_df.drop(columns='date')

# Align data lengths

merged_df['time'] = pd.to_datetime(merged_df['time'])
merged_df = merged_df[merged_df['time'].dt.year != 2024]

#Change the pet unit to millimeters per month

data = merged_df
data['time'] = pd.to_datetime(data['time'])
data.set_index('time', inplace=True)
data['days_in_month'] = data.index.days_in_month
data['pet'] = data['pet'] * data['days_in_month']
csv_path = 'longmen_meteorological_data_1.csv'
data.to_csv(csv_path)

# Create lag features

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from boruta import BorutaPy
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import mutual_info_regression
import joblib
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

data = pd.read_csv('longmen_meteorological_data_1.csv')
original_features = ['cld', 'dtr', 'frs', 'pet', 'pre', 'tmn', 'tmp', 'tmx', 'vap', 'wet']
targets = [col for col in data.columns if 'spei' in col]
lags = list(range(1, 21))

def create_lagged_features(df, features, lags):
    df_lag = df.copy()
    lagged_features = []
    
    for feature in features:
        for lag in lags:
            lag_series = df_lag[feature].shift(lag).rename(f'{feature}_{lag}')
            lagged_features.append(lag_series)
    df_lag = pd.concat([df_lag] + lagged_features, axis=1)
    
    return df_lag
data_with_lags = create_lagged_features(data, original_features, lags)

data_with_lags.to_excel('longmen_meteorological_data_20step.xlsx', index=False, engine='openpyxl')
    